Cleaning data

In [ ]:
import pandas as pd
import unicodedata


nba = pd.read_csv("nba all.csv")

salary_raw = pd.read_csv("salary_dataset.csv", header=None)


salary = salary_raw.iloc[2:, [1, 2]].copy()
salary.columns = ["Player", "Salary"]
salary = salary.dropna(subset=["Player"])
salary["Player"] = salary["Player"].astype(str).str.strip()
salary["Salary"] = pd.to_numeric(salary["Salary"], errors="coerce")


def normalize_name(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.strip().lower()

nba["name_key"] = nba["Player"].apply(normalize_name)
salary["name_key"] = salary["Player"].apply(normalize_name)


matched_nba_rows = nba[nba["name_key"].isin(set(salary["name_key"]))].copy()

matched_nba_rows = matched_nba_rows.merge(
    salary[["name_key", "Salary"]].drop_duplicates("name_key"),
    on="name_key",
    how="left"
)

matched_nba_rows = matched_nba_rows.drop(columns=["name_key"])

print("Matched rows:", len(matched_nba_rows))
matched_nba_rows.head(20)

Matched rows: 274


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Awards,Player-additional,Salary
0,1.0,Shai Gilgeous-Alexander,26.0,OKC,PG,76.0,76.0,2598.0,860.0,1656.0,...,486.0,131.0,77.0,183.0,164.0,2484.0,0.0,MVP-1DPOY-10CPOY-8ASNBA1,gilgesh01,61005000
1,2.0,Anthony Edwards,23.0,MIN,SG,79.0,79.0,2871.0,721.0,1612.0,...,359.0,91.0,51.0,249.0,150.0,2177.0,0.0,MVP-7CPOY-3ASNBA2,edwaran01,45550512
2,3.0,Nikola Jokić,29.0,DEN,C,70.0,70.0,2571.0,786.0,1364.0,...,716.0,127.0,45.0,230.0,160.0,2071.0,34.0,MVP-2CPOY-2ASNBA1,jokicni01,55224526
3,4.0,Giannis Antetokounmpo,30.0,MIL,PF,67.0,67.0,2289.0,793.0,1319.0,...,433.0,58.0,78.0,206.0,155.0,2036.0,11.0,MVP-3DPOY-8ASNBA1,antetgi01,54126450
4,5.0,Jayson Tatum,26.0,BOS,PF,72.0,72.0,2624.0,662.0,1465.0,...,431.0,76.0,38.0,209.0,157.0,1932.0,2.0,MVP-4CPOY-10ASNBA1,tatumja01,54126450
5,6.0,Devin Booker,28.0,PHO,SG,75.0,75.0,2795.0,654.0,1420.0,...,529.0,67.0,16.0,220.0,198.0,1923.0,0.0,NaN,bookede01,53142264
6,7.0,Trae Young,26.0,ATL,PG,76.0,76.0,2739.0,566.0,1376.0,...,880.0,91.0,12.0,355.0,145.0,1841.0,0.0,CPOY-4AS,youngtr01,46394100
7,8.0,Tyler Herro,25.0,MIA,SG,77.0,77.0,2725.0,651.0,1378.0,...,424.0,69.0,17.0,198.0,85.0,1840.0,0.0,AS,herroty01,32500000
8,9.0,Cade Cunningham,23.0,DET,PG,70.0,70.0,2452.0,684.0,1457.0,...,638.0,71.0,53.0,309.0,195.0,1830.0,9.0,MVP-7ASNBA3,cunnica01,46394100
9,10.0,James Harden,35.0,LAC,PG,79.0,79.0,2789.0,531.0,1295.0,...,687.0,118.0,55.0,341.0,162.0,1802.0,3.0,MVP-10ASNBA3,hardeja01,39182693


In [4]:
matched_nba_rows.to_csv("matched_nba_rows.csv", index=False)
print("Saved to matched_nba_rows.csv")

Saved to matched_nba_rows.csv


In [11]:
import pandas as pd
import unicodedata
import os

def norm_name(x):
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.strip().lower()

def read_csv_fallback(path, **kwargs):
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode: {path}")

# Handle filename typo: "matcted_nba_rows.csv" vs "matched_nba_rows.csv"
matched_path = "matcted_nba_rows.csv"
if not os.path.exists(matched_path):
    matched_path = "matched_nba_rows.csv"

matched_df = read_csv_fallback(matched_path)
salary_raw = read_csv_fallback("salary_dataset.csv", header=None)

# Names from matched_nba_rows.csv -> list
matched_names = []
for name in matched_df["Player"]:
    if pd.notna(name) and str(name).strip() != "":
        matched_names.append(norm_name(name))

# Names from salary_dataset.csv -> list
# (player names are column 1, data starts at row index 2)
salary_names = []
for i in range(2, len(salary_raw)):
    name = salary_raw.iloc[i, 1]
    if pd.notna(name) and str(name).strip() != "":
        salary_names.append(norm_name(name))

# Compare
matched_set = set(matched_names)
salary_set = set(salary_names)

missing_in_matched = sorted(salary_set - matched_set)   # in salary, not in matched
extra_in_matched = sorted(matched_set - salary_set)     # in matched, not in salary

print("Unique names in matched:", len(matched_set))
print("Unique names in salary :", len(salary_set))
print("Missing in matched     :", len(missing_in_matched))
print(missing_in_matched)
print("Extra in matched       :", len(extra_in_matched))
print(extra_in_matched)

Unique names in matched: 230
Unique names in salary : 228
Missing in matched     : 11
['alperen azenga1⁄4n', 'bogdan bogdanovia‡', 'dario saric', 'dennis schra¶der', 'jonas valana\x8dia«nas', 'jusuf nurkia‡', 'kristaps porzia†a£is', 'luka doncic', 'nikola jokic', 'nikola jovic', 'nikola vua\x8devia‡']
Extra in matched       : 13
['a.j. green', 'alperen ?engun', 'bogdan bogdanovi?', 'dario ?ari?', 'dennis schroder', 'jonas valan?i?nas', 'jusuf nurki?', 'kristaps porzi??is', 'luka don?i?', 'nikola joki?', 'nikola jovi?', 'nikola vu?evi?', 'toumani camara']
